# Policy Comparison

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn as nn

from battery_sim.battery.battery import Battery
from battery_sim.agents.policies.rule_based import ThresholdPolicy
from battery_sim.agents.policies.probabilistic import ProbabilisticThresholdPolicy
from battery_sim.agents.policies.adp import ADPPolicy, default_features
from battery_sim.agents.policies.mppi import MPPIPolicy
from battery_sim.models.median_reversion import MedianReversionModel
from battery_sim.optimization.policy_optimizer import evaluate_full
from battery_sim.utils.plotting import plot_eval, plot_daily_revenue, compare_policies

In [ ]:
# Load data
train_df = pd.read_csv('../data/NEM_NSW1_2025-01-01_2025-12-31_1h.csv')
test_df = pd.read_csv('../data/NEM_NSW1_2026-01-01_2026-03-01_1h.csv')

print(f'Train: {len(train_df)} rows ({train_df.iloc[0, 0]} to {train_df.iloc[-1, 0]})')
print(f'Test:  {len(test_df)} rows ({test_df.iloc[0, 0]} to {test_df.iloc[-1, 0]})')

Train: 8736 rows (2025-01-01 10:00:00 to 2025-12-31 09:00:00)
Test:  1416 rows (2026-01-01 10:00:00 to 2026-03-01 09:00:00)


In [ ]:
# Battery config
battery = Battery(
    capacity_mwh=2.0,
    max_charge_rate_mw=1.0,
    max_discharge_rate_mw=1.0,
    efficiency=0.9,
)

In [ ]:
Q = 20
vals = np.quantile(train_df.price, q = [i/Q for i in range(1, Q)])
PARAM_GRID = {
    'buy_threshold':vals,
    'sell_threshold':vals,
}

## 1. Deterministic Threshold Policy

In [ ]:
det_policy = ThresholdPolicy(
    buy_threshold=50.0,
    sell_threshold=200.0,
    charge_rate=1.0,
    discharge_rate=1.0,
)

# Learn optimal thresholds on training data
det_policy.learn(
    train_data=train_df, 
    battery=battery, 
    num_iters=25, 
    window_len=24 * 7,
    param_grid=PARAM_GRID
)
print(f'Learned buy_threshold: {det_policy.buy_threshold}')
print(f'Learned sell_threshold: {det_policy.sell_threshold}')

Learned buy_threshold: 109.032085
Learned sell_threshold: 168.311665


## 2. Probabilistic Threshold Policy

In [ ]:
price_model = MedianReversionModel(window_days=30, interval_minutes=60)

prob_policy = ProbabilisticThresholdPolicy(
    model=price_model,
    buy_threshold=50.0,
    sell_threshold=200.0,
    charge_rate=1.0,
    discharge_rate=1.0,
)

# Learn: fits model + grid searches thresholds
prob_policy.learn(
    train_data=train_df, 
    battery=battery, 
    num_iters=25, 
    window_len=24 * 7,
    param_grid=PARAM_GRID
)
print(f'Learned buy_threshold: {prob_policy.buy_threshold}')
print(f'Learned sell_threshold: {prob_policy.sell_threshold}')

Learned buy_threshold: 69.054167
Learned sell_threshold: 109.032085


## 3. ADP (Approximate Dynamic Programming) Policy

In [ ]:
# Single-layer linear value function: 2 features (price, soc) -> 1
adp_net = nn.Linear(2, 1)
adp_policy = ADPPolicy(
    net=adp_net,
    feature_fn=default_features,
    charge_rate=1.0,
    discharge_rate=1.0,
)

adp_policy.learn(
    train_data=train_df,
    battery=battery,
    num_iters=20,
    window_len=24 * 7,
    n_windows=25,
)

# Plot convergence
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(adp_policy.loss_history)
ax.set(xlabel='Iteration', ylabel='MSE Loss', title='ADP Value Function Convergence')
plt.tight_layout()
plt.show()

## 4. MPPI (Model Predictive Path Integral) Policy

In [ ]:
mppi_model = MedianReversionModel(window_days=30, interval_minutes=60)
mppi_policy = MPPIPolicy(
    model=mppi_model,
    battery=battery,
    horizon=12,
    n_samples=100,
    n_trajectories=50,
)

mppi_policy.learn(
    train_data=train_df,
    battery=battery,
    num_iters=10,
    window_len=24 * 7,
)
print(f'Tuned noise_sigma: {mppi_policy.noise_sigma}')
print(f'Tuned temperature: {mppi_policy.temperature}')

## 5. Evaluate on Test Set

In [ ]:
# Re-fit MPPI model on train data before test evaluation
mppi_model.fit(train_df)

det_eval = evaluate_full('Deterministic Threshold', det_policy, test_df, battery)
prob_eval = evaluate_full('Probabilistic Threshold', prob_policy, test_df, battery)
adp_eval = evaluate_full('ADP (Linear)', adp_policy, test_df, battery)
mppi_eval = evaluate_full('MPPI', mppi_policy, test_df, battery)

all_evals = [det_eval, prob_eval, adp_eval, mppi_eval]
for e in all_evals:
    print(e.summary())
    print()

## 6. Detailed Plots

In [ ]:
for e in all_evals:
    plot_eval(e)
    plt.show()

In [ ]:
for e in all_evals:
    plot_daily_revenue(e)
    plt.show()

## 7. Policy Comparison

In [ ]:
compare_policies(all_evals)
plt.show()